# HW 2, Problem 1: How fast can you compute $\pi$?

Humans have been trying to estimate $\pi$ for thousands of years. For example;

- Babylonians (c. 1900 BCE): Approximated $\pi$ as 3.125, {expressed as the fraction $25/8$), which was used for practical construction and geometry.
- Ancient Egyptians (c. 1650 BCE): Documented in the Rhind Mathematical Papyrus, scribe Ahmes estimated $\pi$ to be about 3.1605 (calculated from $(16/9)^2$)
- Archimedes of Syracuse (c. 250 BCE): This Greek mathematician created the first known rigorous **geometric algorithm to bound** $\pi$ by inscribing and circumscribing regular polygons around a circle, proving $(223/71) < \pi < (22/7)$ (roughly, bounding $\pi$ as between 3.1408 and 3.1429).
- The Chinese mathematician used a polygon-based method to calculate $\pi$ accurately to five decimal places: $\pi \approx 3.1416$.
- Zu Chongzhi (c. 480 CE): The Chinese mathematician pushed polygon methods further to show that $3.1415926 < \pi < 3.1415927$, which was then the best estimate available for several hundreds of years.

In the age of computing, there are many ways to estimate $\pi$, and they are **not** equally good. You will write three, time them, and measure how quickly each one gets more accurate (with additional computations). Along the way you will use the vectors, inner products and norms from Unit 2, and get a first taste of random numbers (Unit 3).

**How to use this notebook**
- Run cells from top to bottom with **Shift + Enter**. Fill in every `...`. A message appears after most cells telling you whether it looks right.
- Whenever you see **Predict first**, write your guess before you run the code. Being wrong is fine and is where the learning happens.
- Expect about 30 to 40 minutes.
- At the end, try to break things in Part 3!

In [4]:
import numpy as np, time
import matplotlib.pyplot as plt

STUDENT_ID = 1234567      # <-- put your student ID here: it seeds YOUR random numbers

def median_time(f, *args, repeats=5):
    '''Run f(*args) several times and return the median time in seconds.'''
    times = []
    for _ in range(repeats):
        start = time.perf_counter()
        f(*args)
        times.append(time.perf_counter() - start)
    return float(np.median(times))

def loglog_plot(xs, curves, ylabel, title):
    '''curves is a dict: label -> list of y values (same length as xs).'''
    plt.figure(figsize=(6, 4))
    for label, ys in curves.items():
        plt.loglog(xs, np.maximum(ys, 1e-17), "o-", label=label)   # keeps zeros off the log axis
    plt.xlabel("N (number of terms or points)"); plt.ylabel(ylabel)
    plt.title(title); plt.grid(True, which="both", alpha=0.3); plt.legend(); plt.show()

def check(ok, hint):
    print("OK - looks right!" if ok else "NOT YET: " + hint)

# Part 1: A slow series, written two ways

From https://en.wikipedia.org/wiki/Leibniz_formula_for_%CF%80 :

There is an exact, infinite series representation for $\pi$ that was first discovered by the Indian mathematician Madhava of Sangamagrama (or his followers) in the 14th–15th century, and later independently rediscovered by James Gregory (1671) and Leibniz (1673).

It is usually called the **Leibniz series**, and it says:

$$\frac{\pi}{4} = 1 - \frac13 + \frac15 - \frac17 + \cdots = \sum_{k=0}^{N-1} \frac{(-1)^k}{2k+1} \quad (N \to \infty).$$

**Predict first:** with only the first 4 terms, is your estimate of $\pi$ a bit above or a bit below 3.14? By roughly how much?

**Your prediction:**

## Task 1.1: With a `for` loop
Fill in the line inside the loop: the $k$-th term is `(-1)**k / (2*k + 1)`.

In [5]:
def pi_loop(N):
    total = 0.0
    for k in range(N):
        total += ...          # the k-th term
    return 4 * total

check(abs(pi_loop(4) - 2.8952380952) < 1e-9, "pi_loop(4) should be 2.8952... (the first 4 terms)")

TypeError: unsupported operand type(s) for +=: 'float' and 'ellipsis'

## Task 1.2: With vectors (no loop)
Put the signs $+1, -1, +1, \dots$ in one vector and the numbers $\frac{1}{2k+1}$ in another. Then $\pi/4$ is just their **inner product**.

In [ ]:
def pi_vec(N):
    k = np.arange(N)                 # the vector 0, 1, 2, ..., N-1
    signs = (-1.0) ** k              # +1, -1, +1, ...
    recips = ...                     # the vector of 1/(2k+1)
    return 4 * ...                   # 4 times the inner product of signs and recips

check(abs(pi_vec(4) - pi_loop(4)) < 1e-12 and abs(pi_vec(1000) - pi_loop(1000)) < 1e-9,
      "pi_vec should give the same answer as pi_loop")

## Task 1.3: Race them
**Predict first:** which is faster at $N = 10$? At $N = 1{,}000{,}000$ (i.e., `N=10**6`)? By roughly what factor?

**Your prediction:**

Then run the cell (nothing to fill in). It takes the median of several runs, which is much more reliable than a single run.

In [ ]:
Ns = [10, 100, 1000, 10**4, 10**5, 10**6         # Number of terms in series used to estimate pi
t_loop = [median_time(pi_loop, N) for N in Ns]
t_vec  = [median_time(pi_vec,  N) for N in Ns]
loglog_plot(Ns, {"for loop": t_loop, "vector": t_vec}, "time (s)", "Time to compute pi")

**Question 1:** At about what $N$ do the two lines cross? Which method wins on each side, and roughly by what factor at $N = 10^6$? Why might the vector method lose when $N$ is tiny? (Hint: some of the work required is the same no matter how small $N$ is.)

**Your answer:**

## Task 1.4: How fast does the error shrink?
The error is `abs(estimate - np.pi)`. On a log-log plot a power law $\text{error} \approx C N^{-p}$ is a **straight line with slope $-p$**. Fit a line to the logs to find $p$. (Hint: `np.polyfit(np.log10(Ns), np.log10(errors), 1)[0]` returns the slope.)

In [ ]:
err_leibniz = [abs(pi_vec(N) - np.pi) for N in Ns]
slope = ...                         # slope of the line through (log10 N, log10 error)
print(f"slope = {slope:.2f}")
loglog_plot(Ns, {"Leibniz": err_leibniz}, "error |estimate - pi|", "Leibniz error")
check(abs(slope + 1) < 0.1, "the slope should be close to -1")

**Question 2:** A slope of $-1$ means every 10 times more terms gives only $10^{-1}=\frac{1}{10}$ times the error. (Generalizing, a slope of $k$ means every $m$ times more terms gives $m^k$ times the error.) About how many terms would you need for 10 correct digits? Is Leibniz a good way to compute $\pi$?

**Your answer:**

# Part 2: Two much better ideas

## Task 2.1: Throw darts, randomly (Monte Carlo)
Throw $N$ random darts at the unit square $[0,1]\times[0,1]$. A dart lands inside the quarter circle when its **norm** (distance from the corner) is at most 1. The fraction, $f$, of darts that land inside is about $\pi/4$, since the area of a quarter circle of radius $r$ is exactly $\pi/4$. Below are two examples, where the $x$ and $y$ coordinates are selected from a uniform random distribution:

<img src="pi_est1.png" width="400" height="auto" alt="Darts Example 1"><img src="pi_est2.png" width="400" height="auto" alt="Darts Example 2">

In [ ]:
def pi_monte_carlo(N, rng):
    pts = rng.random((N, 2))           # N random points: an N x 2 matrix
    lengths = ...                      # the norm of each ROW of pts (hint: np.linalg.norm(pts, axis=1))
    return 4 * np.mean(lengths <= 1)

rng = np.random.default_rng(STUDENT_ID)
Ns_mc = [10**2, 10**3, 10**4, 10**5, 10**6]
# average the error over 20 tries per N, because one try is very noisy
err_mc = [np.mean([abs(pi_monte_carlo(N, rng) - np.pi) for _ in range(20)]) for N in Ns_mc]
slope_mc = np.polyfit(np.log10(Ns_mc), np.log10(err_mc), 1)[0]
print(f"slope = {slope_mc:.2f}")
loglog_plot(Ns_mc, {"Monte Carlo (average of 20)": err_mc}, "error |estimate - pi|", "Monte Carlo error")
check(abs(slope_mc + 0.5) < 0.2, "the slope should be close to -0.5")

**Question 3:** The slope is about $-1/2$: to get 10 times more accurate you need **100 times** more darts. Is Monte Carlo better or worse than Leibniz for computing $\pi$? ("Throwing darts, randomly" is *much* more useful for problems where nothing else works! You will see why in Unit 3.)

**Your answer:**

## Task 2.2: A clever series (Machin, 1706)
https://en.wikipedia.org/wiki/Machin-like_formula

Note that: $\pi = 16\arctan\tfrac15 - 4\arctan\tfrac1{239}$,

and we can estimate $\arctan$ with a finite number of terms in the summation below:
$$\qquad \arctan x = x - \frac{x^3}{3} + \frac{x^5}{5} - \cdots = \sum_{k=0}^{n-1} (-1)^k \frac{x^{2k+1}}{2k+1}.$$

This is basically the Leibniz idea again, but with an extra $x^{2k+1}$ that shrinks the terms **very** fast when $x$ is small. Write `arctan_series(x, n_terms)` with the same signs vector as before, then compare it with the other two methods.

In [ ]:
def arctan_series(x, n_terms):
    k = np.arange(n_terms)
    signs = (-1.0) ** k
    terms = ...                      # the vector x**(2k+1) / (2k+1)
    return np.inner(signs, terms)

def pi_machin(n_terms):
    return 16 * arctan_series(1/5, n_terms) - 4 * arctan_series(1/239, n_terms)

check(abs(pi_machin(10) - np.pi) < 1e-12, "pi_machin(10) should already be very close to pi")

n_list = list(range(1, 16))
err_machin = [abs(pi_machin(n) - np.pi) for n in n_list]
loglog_plot(n_list, {"Machin": err_machin}, "error |estimate - pi|", "Machin error")
print("error with 10 terms:", abs(pi_machin(10) - np.pi))

**Question 4:** Compare: how many Leibniz terms, and how many Monte Carlo darts, give the same accuracy that Machin reaches with about 10 terms? The Machin error stops improving at about $10^{-16}$. Why can't it go lower? (_Hint: how many digits does a Python float store?!_)

Python `float` variables follow the IEEE 754 double-precision standard, using 64 bits, each using 1 sign bit, 11 exponent bits, and 52 fraction bits. What is 2^{52}? The accuracy should be on the order of "one part in $2^{52}$".

**Your answer:**

# Part 3: Try to break it!

Each prompt has a tiny piece of code. Run it, then answer in a sentence.

In [ ]:
# Break it #1: memory.  Do NOT run pi_vec(10**10)!  Just compute how big the arrays would be.
N = 10**10
gigabytes_per_array = 8 * N / 1e9          # a float uses 8 bytes
print(f"one array of {N:,} floats = {gigabytes_per_array:,.0f} GB")

**Break it #1:** `pi_vec` makes at least three arrays of length $N$ (i.e., `k`, `signs`, `recips`). Why is $N = 10^{10}$ impossible on a laptop? Name **one** way to still add up $10^{10}$ terms without storing them all at once.

**Your answer:**

In [ ]:
# Break it #2: how reliable is ONE Monte Carlo run?
rng = np.random.default_rng(STUDENT_ID)
print([round(pi_monte_carlo(1000, rng), 3) for _ in range(10)])

**Break it #2:** Ten runs with $N = 1000$ darts each. How much do they differ from each other? Would you trust a single run to give 2 correct digits of $\pi$?

**Your answer:**

In [6]:
# Break it #3: can we get a perfect answer?
print(abs(pi_machin(15) - np.pi))
print(abs(pi_machin(100) - np.pi))
print('Are they exactly the same?',pi_machin(100) == np.pi)

NameError: name 'pi_machin' is not defined

**Break it #3:** Does using 100 terms instead of 15 help? What does that tell you about the limits of computing with floats?

**Your answer:**

# What you found
Fill in the blanks from your plots (a number or a few words each). Later, you we be asked to revisit some of your answers as part of a short online quiz (on Canvas) for Homework 2. 

- Leibniz error slope: **_____** &nbsp;&nbsp; Monte Carlo error slope: **_____**
- The loop and vector timings cross near $N \approx$ **_____**
- Machin reaches about $10^{-16}$ with about **_____** terms.
- One sentence: which method would you use to get 10 digits of $\pi$, and why?

**Before you submit:** use **Kernel → Restart & Run All** and make sure nothing gives an error. Save and submit on Gradescope.